In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "groq:openai/gpt-oss-120b"
)

## Summarization Middleware

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),
            keep=("messages",4)

        )
    ]
)

In [3]:
config={"configurable":{"thread_id":"test-1"}}

In [4]:
questions = [
    "What is 2+2",
    "what is 10*5",
    "what is 100/4",
    "what is 15-7",
    "what is 20+36",
    "what is 4*4"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='1c8e4cf4-d524-41f8-bdeb-6ebeceda4eea'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks "What is 2+2". Simple. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 77, 'total_tokens': 115, 'completion_time': 0.079213964, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.003210859, 'prompt_tokens_details': None, 'queue_time': 0.199595438, 'total_time': 0.082424823}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4b2f03d631', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2257-fb9b-7a01-b08b-3c5ec0ada64a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 38, 'total_tokens': 115, 'output_token_details': {'reasoning': 19}})]}
Messages: 2
Mes

### Token Size

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [6]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~358 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='b58472c6-8a28-4765-b66a-b6c42dbaa317'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants hotels in Paris. We have a function search_hotels(city). We should call it.', 'tool_calls': [{'id': 'fc_8b3eadda-5acf-44b4-8bd6-49eea817542b', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 129, 'total_tokens': 179, 'completion_time': 0.104180766, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.005149988, 'prompt_tokens_details': None, 'queue_time': 0.234357331, 'total_time': 0.109330754}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d63b2695ba', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2258-2670-75d1-b65

## Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

## Human in the Loop

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)-> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [9]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approved","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [11]:
config = {"configurable":{"thread_id":"test-approve"}}
#step 1
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [12]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b843c9ca-9dad-41e7-857d-3f8e2f3de765'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. We have a function send_email_tool. Use it.', 'tool_calls': [{'id': 'fc_f743759b-fa58-4271-bcaa-eea03f9f9868', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 174, 'total_tokens': 240, 'completion_time': 0.140430794, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.008444608, 'prompt_tokens_details': None, 'queue_time': 0.260359833, 'total_time': 0.148875402}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'l

In [ ]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")